In [1]:
import sys

if sys.version_info[0] + 0.1 * sys.version_info[1] < 3.6:
    raise Exception("Must be using Python 3.6+ version")

In [2]:
%matplotlib inline
sys.path.append("..")

from plotly.subplots import make_subplots
import plotly.graph_objects as go

import numpy as np

import wavelet_transform.utils.training_data as train
from wavelet_transform.postprocess import plotting as plot
import wavelet_transform.utils.ACD_data as acd
import wavelet_transform.wpt.inverse_transform as inv
from wavelet_transform.utils.extensions import fft, psd, to_dB, eng_form_plot, trim_data
from wavelet_transform.utils.machine_learning import fit_and_evaluate, train_k_means, assess_parameter
import wavelet_transform.utils.kmeans_results as results
from wavelet_transform.utils.preprocessing import preprocess
from wavelet_transform.utils.save_plots import save_figure

result_id is used to store the results in a file that can be found using results.get_data(result_id)

In [3]:
saveflag = False

In [6]:
frequency = 1.1e6
voltage = 100
duty_cycle = 5
exp_no = 1
use_acd_data = False  # set to False to ues pcd data
data, t_plot = acd.get_data(voltage, exp_no, duty_cycle, use_acd_data) 

no_clusters = (3, 6)
# result_id = f"{voltage}_{pcd_number}_{no_clusters}"

In [ ]:
sampling_rate = np.round(1 / (t_plot[1] - t_plot[0]))
# freq_threshold = 20e6  # the nodes with norm_freq_range above this threshold will not be kept
# freq_threshold /= (sampling_rate / 2)
freq_threshold = 1
print(sampling_rate)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_plot, y=data, mode="lines"))
fig.update_layout(
    title="data",
    xaxis_title="time/s",
    yaxis_title="voltage/Pa"
)

In [ ]:
# Parameters for the wavelet transform
wavelet = "db30" # "coif17", ""
levels = 12  # set to None to use max-level transform
mode = "symmetric"
use_min_entropy = True
min_level = 7

In [ ]:
# Normalise
data_processed = preprocess(data, window=False)

In [ ]:
# # Trim the array down to a start and end time
t_lims = (100e-6, 170e-6)
data_trim, t_plot = trim_data(data, t_plot, t_lims, return_time=True)
data_processed = preprocess(data_trim)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_plot, y=data_trim, mode="lines"))
fig.update_layout(
    title="data",
    xaxis_title="time/s",
    yaxis_title="voltage/Pa"
)

In [ ]:
# Plot generated data
# fft_data, freq = fft(data_processed, sampling_rate, frequency)
fft_lim = 10
def plot_fft(data, sampling_rate, frequency):
    fft_data, freq = fft(data, sampling_rate, frequency)
    
    # plot sound wave and fft
    fig = make_subplots(rows=1, cols=1)

    fig.add_trace(
        go.Scatter(x=freq, y=to_dB(np.abs(fft_data)), mode="lines", name="fft - amp"),
        row=1,
        col=1,
    )
    
    fig.update_layout(
        height=500,
        width=1000,
        font_size=20,
        xaxis1=dict(title="Freq [f/f<sub>0</sub>]", range=[0, fft_lim]),
        yaxis1=dict(title="Amplitude [dB]", range=[-40, 40]), 
        legend=dict(x=0.8, y=1, font=dict(size=20)),
        plot_bgcolor="white",     
    )
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey')        
    fig.show()
    return fig

In [ ]:
fig = plot_fft(data_processed, sampling_rate, frequency)
if saveflag is True:
    save_figure(fig, f"{result_id}_data")

In [ ]:
from scipy import signal
import matplotlib.pyplot as plt

def stft_plt(data, frequency, period, sampling_rate, tick_size, fsize):
    fig, ax = plt.subplots()
    fig.set_size_inches(18, 9)
    f_stft, t_stft, mag_stft = signal.stft(data, sampling_rate, nperseg=2**11, detrend=False, scaling='spectrum')
    stft_plt = ax.pcolormesh(t_stft/period, f_stft/frequency, 20*np.log10(np.abs(mag_stft)), vmin=-100, vmax=1, shading='gouraud')
    ax.set_ylabel("$f / f_0$ [DL]", fontsize=fsize)
    ax.set_xlabel("t/T [DL]", fontsize=fsize)  # noqa
    ax.set_title('STFT - spectrum', fontsize=fsize)
    ax.set_ylim([0,10]) 
    plt.xticks(fontsize=tick_size)
    plt.yticks(np.arange(0,11),np.arange(0,11),fontsize=tick_size)
    cbar_label = ("Magnitude [dB]")
    fig.tight_layout()
    cbar = fig.colorbar(stft_plt, pad=0.01)
    cbar.ax.tick_params(labelsize=tick_size)
    cbar.set_label(label=cbar_label, size=fsize)
    plt.show()

In [ ]:
stft_plt(data_processed, frequency, 1/frequency, sampling_rate, 16, 20)

## Wavelet transform

In [ ]:
wpt_df, tr_obj = train.generate_training_data(
    data_processed, levels, wavelet, mode, 
    include_tr_obj=True, min_entropy=use_min_entropy,
    min_level=min_level, freq_threshold=freq_threshold
)

## Training

In [ ]:
# k-means parameters
kmeans_params = {
    "init": "k-means++",
    "n_init":10,
    "max_iter":100,
    "tol":1e-8,
    "verbose":0,
    "algorithm":"lloyd",
}

# training data
classifying_metrics = [
    "log average energy",
    "kurtosis",
    # "crest factor",
    # "level entropy",
    "spectral entropy",
    "average energy",
    # "skewness"
]
# Set to no_clusters to None to select optimal number of clusters by silhouette score
use_inverse_metrics = False
scale_data = "standard"
if use_inverse_metrics is True:
    inv_tr_obj = inv.InverseTransform(tr_obj)  # Use info from inverse transform object instead
    wpt_df_train = inv_tr_obj.info(leaf_nodes=True, freq_threshold=freq_threshold)
else:
    wpt_df_train = wpt_df.copy()

In [ ]:
data_train, labels, data_train_normalised, centers, scaler, kmeans = train_k_means(
    wpt_df_train,
    classifying_metrics,
    no_clusters=no_clusters,
    kmeans_params=kmeans_params,
    scale_data=scale_data
)

In [ ]:
ordered_centers, ordered_labels = plot.relabel(labels, centers, "average energy")

## Save results

In [ ]:
if saveflag == True:
    additional_params = {
        "raw_data": data,
        "driving frequency": frequency,
    }

    results.save_data(
        result_id, data_train, tr_obj, centers, labels, 
        sampling_rate, freq_threshold, **additional_params
    )

## Plot results

In [ ]:
fig = plot.plot_clustering(
    data_train, 
    ordered_centers,
    ordered_labels, 
    [classifying_metrics[i] for i in [0, 1, 2]], 
    title=None
)

In [ ]:
# Plot centroid metrics
# xaxis_ticks_labels = dict((i, f"Clu÷ster {i}") for i in range(len(centers)))
fig = plot.plot_centers(classifying_metrics, ordered_centers) #, label_map=xaxis_ticks_labels)

In [ ]:
tr_obj.calc_frequencies(sampling_rate)
fig = plot.heisenberg(
    tr_obj, t_plot[-1], max_freq=freq_threshold/4, padding="repeat", use_log_scale=True,
    vmin=-2, vmax=0, color_map="viridis"
)

# tr_obj.plot_tree(
#     node_size="average energy",
#     node_colour=list(labels), 
#     freq_threshold=freq_threshold, 
#     order="natural",
#     sampling_rate=sampling_rate,
#     show_frequency=False,
# )

In [ ]:
# Set clusters to None to plot all of them, otherwise, plot a list of int
fig = plot.plot_fft_by_group(
    tr_obj, labels, sampling_rate,
    clusters=None,  fundamental_freq=frequency,
    freq_threshold=freq_threshold, fig_height=350,
    plot_full_reconstruction=False    
)

fig = plot.plot_reconstructions_by_group(
    tr_obj, labels, t_plot, freq_threshold
)

In [ ]:
from wavelet_transform.postprocess.plotting import get_reconstructions_by_group
import plotly.graph_objects as go
recs = get_reconstructions_by_group(tr_obj, labels, freq_threshold)
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=t_plot,
        y=sum(recs),
        mode="lines",
        name="Signal with freq thresholding",
    )
)
fig.add_trace(
    go.Scatter(
        x=t_plot,
        y=tr_obj.data,
        mode="lines",
        name="Orignial signal"
    )
)


In [ ]:
import wavelet_transform.wpt.inverse_transform as inv
import matplotlib.pyplot as plt

# Add number to plot_centres to plot reconstructions of all nodes belonging to the cluster
plot_centres = []
for i, (node, assignment) in enumerate(zip(tr_obj.get_leaf_nodes(freq_threshold=freq_threshold), labels)):
    if assignment in plot_centres:
        print("spectral entropy:", node.spectral_entropy)
        rec = inv.inv_dwpt(tr_obj, node.level, nodes=node)
        psd_rec, freq = psd(rec, sampling_rate, frequency)
        # freq /= frequency
        plt.plot(frequency, to_dB(psd_rec))
        plt.xlim(0, 10)
        plt.title(f"frequency range: {np.divide(node.frequency_range,frequency)} (normalised by f0)")
        plt.show()
        plt.plot(rec)
        plt.show()

In [ ]:
# [phi, psi, x] = pywt.Wavelet('db2').wavefun(level=4)

In [ ]:
# import numpy as np
# import scipy.signal as sig
# win_func = sig.windows.blackmanharris(len(data_segments[2]), sym=False)

In [ ]:
# np.std(data_segments[2]*win_func)